In [ ]:
# SFT tunes on <query> <response>
# PAUSE token tunes on <query> <pause_token> x K <response>
# where <pause_token> is a special token added to the vocab.
#
# Reference: Goyal et al. (2023) "Think before you speak:
#   Training Language Models With Pause Tokens"
#   https://arxiv.org/abs/2310.02226
#
# Key design choices:
#   - K pause tokens injected between query and response at TRAINING time.
#   - Loss is computed on response tokens only (query + pause tokens masked).
#   - At EVAL time, K pause tokens are prepended after the query prompt,
#     then the model generates freely (greedy / sampling).
#   - Pause token embedding is initialised to the mean of existing embeddings.

In [1]:
import sys
import torch
import torch.nn as nn
from torch.nn.utils.rnn import pad_sequence
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModelForCausalLM
from datasets import load_dataset
import re

sys.path.insert(0, "/Users/fangyuanyu/Implementation/mod_gpt")

# ---- Config ----
MODEL_NAME  = "Qwen/Qwen3-0.6B"
K_PAUSE     = 8        # number of pause tokens inserted between query and response
MAX_LENGTH  = 512      # max total sequence length (query + K_PAUSE + response)
LR          = 1e-5
BATCH_SIZE  = 4
STEPS       = 200      # demo steps
SEED        = 42

torch.manual_seed(SEED)
print(f"K_PAUSE={K_PAUSE}, MODEL={MODEL_NAME}, MAX_LENGTH={MAX_LENGTH}")

/Users/fangyuanyu/anaconda3/lib/python3.11/site-packages/torch/utils/_pytree.py:185: FutureWarning: optree is installed but the version is too old to support PyTorch Dynamo in C++ pytree. C++ pytree support is disabled. Please consider upgrading optree using `python3 -m pip install --upgrade 'optree>=0.13.0'`.
  warnings.warn(


K_PAUSE=8, MODEL=Qwen/Qwen3-0.6B, MAX_LENGTH=512


In [2]:
# ---- A. Load tokenizer + model ----
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
if tokenizer.pad_token_id is None:
    tokenizer.pad_token_id = tokenizer.eos_token_id

model = AutoModelForCausalLM.from_pretrained(MODEL_NAME, torch_dtype=torch.float32)
orig_vocab_size = model.config.vocab_size
print(f"Loaded {MODEL_NAME}  |  vocab={orig_vocab_size}")

# ---- B. Add <pause> special token ----
tokenizer.add_special_tokens({"additional_special_tokens": ["<pause>"]})
PAUSE_ID = tokenizer.convert_tokens_to_ids("<pause>")
print(f"<pause> token id = {PAUSE_ID}")

# ---- C. Resize embeddings, init new row to mean of existing ----
model.resize_token_embeddings(len(tokenizer))

with torch.no_grad():
    emb = model.get_input_embeddings().weight
    mean_emb = emb[:orig_vocab_size].mean(dim=0)
    emb[PAUSE_ID] = mean_emb
    # Tie lm_head if it shares weight with embedding
    if model.get_output_embeddings().weight.data_ptr() != emb.data_ptr():
        model.get_output_embeddings().weight[PAUSE_ID] = mean_emb

print(f"Embedding resized to {len(tokenizer)}  |  <pause> row initialised to embedding mean")

`torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/311 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie model.embed_tokens.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


Loaded Qwen/Qwen3-0.6B  |  vocab=151936
<pause> token id = 151669
Embedding resized to 151670  |  <pause> row initialised to embedding mean


In [4]:
# ---- D. Sequence preparation ----
# Training sequence layout:
#
#   [ query_ids ]  [ pause_id * K ]  [ response_ids ]
#   <-- masked -->  <-- masked -->   <-- loss here -->
#
# The response tokens are the *only* tokens contributing to the CE loss.

def prepare_sample(prompt: str, full_text: str, tokenizer, pause_id: int,
                   k_pause: int, max_length: int):
    """
    Build a pause-augmented input for causal LM training.

    Returns dict with:
        input_ids     : (seq_len,) long tensor
        attention_mask: (seq_len,) long tensor
        labels        : (seq_len,) long tensor  (-100 for query + pause tokens)
        prompt_len    : int  (length of query + pause prefix, for eval convenience)
    """
    query_ids = tokenizer(prompt, add_special_tokens=False)["input_ids"]
    full_ids  = tokenizer(full_text, add_special_tokens=False)["input_ids"]

    # response_ids = tokens that follow the query in full_text
    response_ids = full_ids[len(query_ids):]

    # Truncate response so total length fits in max_length
    max_resp = max_length - len(query_ids) - k_pause
    response_ids = response_ids[:max(max_resp, 1)]

    # Build full sequence
    seq = query_ids + [pause_id] * k_pause + response_ids
    seq = seq[:max_length]

    input_ids = torch.tensor(seq, dtype=torch.long)
    attention_mask = torch.ones_like(input_ids)

    # Labels: mask query + pause prefix
    prefix_len = min(len(query_ids) + k_pause, len(seq))
    labels = input_ids.clone()
    labels[:prefix_len] = -100

    return {
        "input_ids": input_ids,
        "attention_mask": attention_mask,
        "labels": labels,
        "prompt_len": prefix_len,   # points to start of response in seq
    }

# ---- Quick sanity check ----
_tok = tokenizer
_ex  = {"question": "What is 3 + 5?", "answer": "#### 8"}
_prompt   = f"Question: {_ex['question']}\nAnswer:"
_fulltext  = f"{_prompt} {_ex['answer']}"
_sample = prepare_sample(_prompt, _fulltext, _tok, PAUSE_ID, K_PAUSE, MAX_LENGTH)

print("input_ids :", _sample["input_ids"].tolist())
print("labels    :", _sample["labels"].tolist())
print("prompt_len:", _sample["prompt_len"])
print("decoded   :", tokenizer.decode(_sample["input_ids"], skip_special_tokens=False))

input_ids : [14582, 25, 3555, 374, 220, 18, 488, 220, 20, 5267, 16141, 25, 151669, 151669, 151669, 151669, 151669, 151669, 151669, 151669, 26274, 220, 23]
labels    : [-100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, 26274, 220, 23]
prompt_len: 20
decoded   : Question: What is 3 + 5?
Answer:<pause><pause><pause><pause><pause><pause><pause><pause> #### 8


In [5]:
# ---- E. Dataset ----

class PauseTokenDataset(Dataset):
    """GSM8K with pause tokens injected between query and response."""

    def __init__(self, split="train", tokenizer=None, pause_id=None,
                 k_pause=8, max_length=512):
        self.hf_data   = load_dataset("gsm8k", "main", split=split)
        self.tokenizer = tokenizer
        self.pause_id  = pause_id
        self.k_pause   = k_pause
        self.max_length = max_length

    def __len__(self):
        return len(self.hf_data)

    def __getitem__(self, idx):
        ex = self.hf_data[idx]
        prompt    = f"Question: {ex['question']}\nAnswer:"
        full_text = f"{prompt} {ex['answer']}"
        return prepare_sample(
            prompt, full_text, self.tokenizer,
            self.pause_id, self.k_pause, self.max_length,
        )

    @staticmethod
    def extract_answer(text):
        """Extract final numeric answer from GSM8K format: #### <number>"""
        m = re.search(r"####\s*(-?[\d,]+\.?\d*)", text)
        return m.group(1).replace(",", "").strip() if m else None


def pause_collate_fn(batch, pad_token_id):
    input_ids  = [b["input_ids"]  for b in batch]
    labels     = [b["labels"]     for b in batch]
    prompt_lens = [b["prompt_len"] for b in batch]

    padded_ids  = pad_sequence(input_ids, batch_first=True, padding_value=pad_token_id)
    padded_lbls = pad_sequence(labels,    batch_first=True, padding_value=-100)
    attn_mask   = (padded_ids != pad_token_id).long()

    return {
        "input_ids":      padded_ids,
        "attention_mask": attn_mask,
        "labels":         padded_lbls,
        "prompt_lens":    torch.tensor(prompt_lens, dtype=torch.long),
    }


from functools import partial

train_ds = PauseTokenDataset("train", tokenizer, PAUSE_ID, K_PAUSE, MAX_LENGTH)
val_ds   = PauseTokenDataset("test",  tokenizer, PAUSE_ID, K_PAUSE, MAX_LENGTH)

collate = partial(pause_collate_fn, pad_token_id=tokenizer.pad_token_id)
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,
                          collate_fn=collate, num_workers=0)

print(f"Train: {len(train_ds)}  |  Val: {len(val_ds)}")
batch = next(iter(train_loader))
print("input_ids shape :", batch["input_ids"].shape)
print("labels shape    :", batch["labels"].shape)
print("% unmasked (loss tokens):",
      (batch["labels"] != -100).float().mean().item())

Train: 7473  |  Val: 1319
input_ids shape : torch.Size([4, 209])
labels shape    : torch.Size([4, 209])
% unmasked (loss tokens): 0.4449760913848877


In [ ]:
# the very last token is the <pause token>
# it's appended to the prompt

Embedding(151670, 1024)

In [ ]:
input_ids = batch["input_ids"][0]
batch["attention_mask"][0][input_ids == 151669] # attention mask on pause token is 1
# -> but we need to mask out the loss on <pause token> right? 
batch["labels"][0][input_ids == 151669] # -> ok, pause token got their lables masked out already


tensor([-100, -100, -100, -100, -100, -100, -100, -100])

In [6]:
# ---- F. Minimal SFT training loop ----

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model  = model.to(device)

optimizer = torch.optim.AdamW(model.parameters(), lr=LR)

model.train()
step = 0
log_every = 20

for epoch in range(1):          # single epoch for demo; increase for real run
    for batch in train_loader:
        if step >= STEPS:
            break

        input_ids      = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels         = batch["labels"].to(device)

        outputs = model(input_ids=input_ids,
                        attention_mask=attention_mask,
                        labels=labels)
        loss = outputs.loss
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        optimizer.zero_grad(set_to_none=True)
        step += 1

        if step % log_every == 0:
            print(f"step {step:>4}/{STEPS}  loss={loss.item():.4f}")

    if step >= STEPS:
        break

print("Training done.")

KeyboardInterrupt: 

In [ ]:
# ---- G. Evaluation with pause tokens ----
# At inference time, prepend K pause tokens after the query so the model
# gets the same "thinking budget" it was trained with.
#
# Sequence fed to model.generate():
#   [ query_ids ]  [ pause_id * K ]
#   ---> model continues from here (generates response tokens)

@torch.no_grad()
def greedy_generate_with_pause(prompt: str, model, tokenizer,
                                pause_id: int, k_pause: int,
                                max_new_tokens: int = 128):
    """Generate response given a query, prepending K pause tokens."""
    query_ids   = tokenizer(prompt, add_special_tokens=False,
                            return_tensors="pt")["input_ids"]
    pause_block = torch.full((1, k_pause), pause_id, dtype=torch.long)
    input_ids   = torch.cat([query_ids, pause_block], dim=1).to(device)

    out = model.generate(
        input_ids=input_ids,
        max_new_tokens=max_new_tokens,
        do_sample=False,
        pad_token_id=tokenizer.pad_token_id,
    )
    # Decode only the newly generated tokens (after the prefix)
    gen_ids  = out[0, input_ids.size(1):]
    response = tokenizer.decode(gen_ids, skip_special_tokens=True)
    return response


@torch.no_grad()
def evaluate_gsm8k(model, tokenizer, val_ds, pause_id, k_pause,
                   num_samples=100, max_new_tokens=128):
    model.eval()
    correct, total = 0, 0
    for i in range(min(num_samples, len(val_ds.hf_data))):
        ex        = val_ds.hf_data[i]
        prompt    = f"Question: {ex['question']}\nAnswer:"
        full_text = f"{prompt} {ex['answer']}"
        gold      = val_ds.extract_answer(full_text)

        response  = greedy_generate_with_pause(
            prompt, model, tokenizer, pause_id, k_pause, max_new_tokens
        )
        pred = val_ds.extract_answer(response)

        hit = (gold is not None and pred is not None
               and pred.strip() == gold.strip())
        correct += int(hit)
        total   += 1

        if (i + 1) % 20 == 0:
            print(f"  [{i+1}/{num_samples}]  acc={correct/(i+1)*100:.1f}%")

    acc = correct / max(total, 1)
    print(f"\nFinal acc: {correct}/{total} = {acc*100:.1f}%")
    model.train()
    return acc

# ---- Run a quick eval on 50 samples ----
model.eval()
acc = evaluate_gsm8k(model, tokenizer, val_ds, PAUSE_ID, K_PAUSE,
                     num_samples=50, max_new_tokens=128)